In [1]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 72.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 33.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.1.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [2]:
import os
import torch
import pandas as pd
import numpy as np
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

In [6]:
class Config:
    MOVIES_PATH = r'/kaggle/input/d/rounakbanik/the-movies-dataset/movies_metadata.csv'
    RATINGS_PATH = r'/kaggle/input/d/rounakbanik/the-movies-dataset/ratings.csv'
    
 
    MODEL_DIRS = [
        "/kaggle/input/gemma-2/transformers/gemma-2-9b-it/2",
        "/kaggle/input/gemma-2/transformers/gemma-2-9b-it/1"
    ]
    MODEL_ID = next((p for p in MODEL_DIRS if os.path.exists(p)), "google/gemma-2-9b-it")
    
    TOP_MOVIES = 1500  
    MIN_VOTES = 20     
    MAX_HIST = 15

In [7]:
def load_and_prep_data():
    
    cols = ['id', 'title', 'genres', 'overview', 'vote_count']
    movies = pd.read_csv(Config.MOVIES_PATH, low_memory=False, usecols=cols)
    movies = movies[pd.to_numeric(movies['id'], errors='coerce').notnull()]
    movies['id'] = movies['id'].astype(int)
    movies['vote_count'] = pd.to_numeric(movies['vote_count'], errors='coerce').fillna(0)
    
    top_movies = movies.sort_values('vote_count', ascending=False).head(Config.TOP_MOVIES).copy()
    
    def format_movie(row):
        try:
            genres = [x['name'] for x in eval(row['genres'])][:3]
            g_str = ", ".join(genres)
        except:
            g_str = "Unknown"
        return f"{row['title']} ({g_str})"

    top_movies['llm_text'] = top_movies.apply(format_movie, axis=1)
    
    ratings = pd.read_csv(Config.RATINGS_PATH)
    ratings = ratings[ratings['movieId'].isin(top_movies['id'])]
    ratings = ratings[ratings['rating'] >= 3.8] 
    
    user_history = ratings.groupby('userId')['movieId'].apply(list).reset_index()
    user_history = user_history[user_history['movieId'].apply(len) >= Config.MIN_VOTES]
    
    movie_map = top_movies.set_index('id')['llm_text'].to_dict()
    
    def get_history_str(m_ids):
        return "\n".join([f"- {movie_map[m]}" for m in m_ids[-Config.MAX_HIST:] if m in movie_map])
        
    user_history['history_str'] = user_history['movieId'].apply(get_history_str)
    
    return user_history

In [9]:
df_users = load_and_prep_data()

In [11]:
def load_llm():
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        Config.MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        local_files_only=os.path.exists(Config.MODEL_ID) 
    )
    
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=400,
        temperature=0.7,
        return_full_text=False
    )
    
    return pipe, tokenizer

In [12]:
llm_pipe, tokenizer = load_llm()

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [14]:
def query_llm(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipe(prompt)
    return out[0]['generated_text'].strip()

In [ ]:
target = df_users.sample(1).iloc[0]
user_id = target['userId']
history = target['history_str']

print(f"USER {user_id} HISTORY:\n{history}\n")


msg_rec = [{"role": "user", "content": f"""
You are a movie expert.
User's watch history:
{history}

Recommend 5 NEW movies (not in history) based on this taste.
Format:
1. Title - Reason
2. Title - Reason
3. Title - Reason
4. Title - Reason
5. Title - Reason
"""}]
print(query_llm(msg_rec))


candidates = df_users[df_users['userId'] != user_id].sample(1)

for _, cand in candidates.iterrows():
    cand_id = cand['userId']
    cand_hist = cand['history_str']
    
    msg_sim = [{"role": "user", "content": f"""
    Compare User A and User B.
    
    User A (Target):
    {history}
    
    User B (Candidate):
    {cand_hist}
    
    Analyze similarity (0-10). If Score >= 7, recommend 5 NEW movies from User B to User A.
    Output format:
    1. Tittle - score:[X/10] - reason:(short text) 
    2. Tittle - score:[X/10] - reason:(short text) 
    3. Tittle - score:[X/10] - reason:(short text) 
    4. Tittle - score:[X/10] - reason:(short text) 
    5. Tittle - score:[X/10] - reason:(short text) 

    """}]
    
    res = query_llm(msg_sim)
    print(f" Comparing with User {cand_id} ")
    print(res)

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER 6697 HISTORY:
- Insomnia (Crime, Mystery, Thriller)
- Magnolia (Drama)
- A Nightmare on Elm Street (Horror)
- Love Actually (Comedy, Romance, Drama)
- Notting Hill (Romance, Comedy, Drama)
- Terminator Salvation (Action, Science Fiction, Thriller)
- The Terminal (Comedy, Drama)
- Men in Black II (Action, Adventure, Comedy)
- The Last Samurai (Drama, Action, War)
- Grease (Romance)
- The Usual Suspects (Drama, Crime, Thriller)
- Meet the Fockers (Comedy, Romance)
- Tomorrow Never Dies (Adventure, Action, Thriller)
- The Evil Dead (Horror)
- From Hell (Horror, Mystery, Thriller)



Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on your eclectic taste, here are 5 movie recommendations:

1. **Se7en (Crime, Mystery, Thriller)** - Like *Insomnia* and *The Usual Suspects*, this film dives into the dark side of crime with a complex mystery and unsettling atmosphere. 
2. **The Silence of the Lambs (Horror, Crime, Thriller)** -  If you enjoyed the horror elements of *A Nightmare on Elm Street* and *The Evil Dead*, this classic thriller with chilling performances will keep you on the edge of your seat.
3. **Heat (Crime, Drama, Action)** -  This film blends the crime and action elements of *Terminator Salvation* and *The Usual Suspects* with a compelling story about two adversaries, a meticulous detective and a master thief, going head-to-head.
4. **Fight Club (Drama, Thriller)** - Combining the social commentary of *Magnolia* with the psychological thriller aspects of *Insomnia*, *Fight Club* offers a thought-provoking and often disturbing journey into masculinity and rebellion. 
5. **Moulin Rouge! (Romance, Mus